In [1]:
#!pip install -r requirements.txt

In [2]:
import torch
from torch.utils.data import DataLoader
import optuna
import numpy as np
import pandas as pd
import feature_engineering
from dataset.dataset_construction import TimeSeriesDataset
from model import *

device = 'cuda' if torch.cuda.is_available() else 'cpu'

c:\Users\jvang\OneDrive\Desktop\UFRGS\6 Semestre\Ciência de Dados\guri-teimoso\Guri-Teimoso\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
set_seed(42)

In [4]:
file_path = './dataset/dataset.csv'
raw_dataset_df = pd.read_csv(file_path)

df_hybrid = feature_engineering.featureEng(raw_dataset_df)

X_train, y_train, X_test, y_test = feature_engineering.createSplit(df_hybrid)

preprocessor = feature_engineering.createPreprocessingPipeline(X_train.columns)

In [5]:
optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress per-trial noise

def objective(trial):
    set_seed(42)

    hp = {
        'input_size':  29,
        'output_size': 1,
        'label_width': 1,
        'hidden_size': trial.suggest_categorical('hidden_size', [64, 128, 256]),
        'num_layers':  trial.suggest_int('num_layers', 1, 5),
        'dropout':     trial.suggest_float('dropout', 0.0, 0.4, step=0.1),
    }

    tp = {
        'batch_size': trial.suggest_categorical('batch_size', [32]),
        'lr':         trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'epochs':     20,
    }

    # Refit preprocessor cleanly for each trial
    trial_preprocessor = feature_engineering.createPreprocessingPipeline(X_train.columns)

    cv_history = crossValidate(
        X_train, y_train,
        hyperparameters=hp,
        preprocessor=trial_preprocessor,
        train_params=tp,
        n_splits=5,
        patience=5,
    )

    # Objective: average best MSE across folds
    mean_mse = np.mean([np.min(h['val_mae']) for h in cv_history])
    return mean_mse

In [6]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f"\nBest MSE : {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

  0%|          | 0/20 [00:01<?, ?it/s]


[W 2026-06-18 17:44:27,051] Trial 0 failed with parameters: {'hidden_size': 256, 'num_layers': 5, 'dropout': 0.2, 'batch_size': 32, 'lr': 0.00013807742528456718} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\jvang\OneDrive\Desktop\UFRGS\6 Semestre\Ciência de Dados\guri-teimoso\Guri-Teimoso\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\jvang\AppData\Local\Temp\ipykernel_29776\1019212795.py", line 24, in objective
    cv_history = crossValidate(
                 ^^^^^^^^^^^^^^
  File "c:\Users\jvang\OneDrive\Desktop\UFRGS\6 Semestre\Ciência de Dados\guri-teimoso\Guri-Teimoso\src\model\model.py", line 60, in crossValidate
    metrics    = evaluate(model, val_loader, device)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jvang\OneDrive\Desktop\UFRGS\6 Semestre\Ciência de Dados\guri-teimoso\Guri-T

KeyboardInterrupt: 

In [8]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

# Drop the NaN rows created by lag shifts
X_train_processed = X_train_processed.dropna()
y_train = y_train.loc[X_train_processed.index]

X_test_processed = X_test_processed.dropna()
y_test = y_test.loc[X_test_processed.index]

train_df = pd.concat([X_train_processed, y_train], axis=1)
test_df  = pd.concat([X_test_processed,  y_test],  axis=1)

train_dataset = TimeSeriesDataset(train_df, input_width=7, label_width=1, shift=1, label_columns=['internacoes'])
test_dataset = TimeSeriesDataset(test_df, input_width=7, label_width=1, shift=1, label_columns=['internacoes'])

train_loader =  DataLoader(train_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [9]:
hyperparameters = {
    'input_size': 29,
    'hidden_size': 64,
    'num_layers': 2,
    'output_size': 1,
    'label_width': 1,
    'dropout': 0.1
}

train_params = {
    'batch_size': 32,
    'lr': 1e-3,
    'epochs': 20
}

model = LSTMModel(
    input_size=hyperparameters['input_size'],
    hidden_size=hyperparameters['hidden_size'],
    num_layers=hyperparameters['num_layers'],
    output_size=hyperparameters['output_size'],
    label_width=hyperparameters['label_width'],
    dropout=hyperparameters['dropout']
)

model.to(device)

LSTMModel(
  (lstm): LSTM(29, 64, num_layers=2, batch_first=True, dropout=0.1)
  (dropout): Dropout(p=0.1, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)

In [10]:
model, history = train(model, train_loader, train_params)
result = evaluate(model, test_loader, device)
print(result)

{'loss': np.float64(4.650453758239746), 'mse': 4.83958625793457, 'mae': 1.6971322298049927, 'rmse': np.float64(2.199905965702755), 'r2': 0.3132135272026062}
